# VinDr-Mammo: **Benign vs Malignant** — Pipeline 3: Oversampling (WeightedRandomSampler)

### Konfigurasi Pipeline 3:
- **Task**: Binary Classification — Benign (0) vs Malignant (1)
- **Label mapping**:
  - `Benign (0)` → BI-RADS 1, 2
  - `Malignant (1)` → BI-RADS 3, 4, 5
- **Data**: Original (imbalanced, tanpa modifikasi)
- **Sampler**: `WeightedRandomSampler` — setiap sample ditarik proporsional terhadap invers frekuensi kelasnya
- **Loss**: `CrossEntropyLoss()` — tanpa pos_weight (balancing dilakukan di sampler)
- **Augmentasi**: HFlip, VFlip, Rotation, GaussianBlur, ColorJitter (standar)
- **Output head**: `NUM_CLASSES=2`, output `[B, 2]` logit → softmax → probability
- **Perbedaan dari P2**: Imbalance ditangani di **level data** (sampler), bukan di level loss

## 1. Install & Import

In [ ]:
!pip install timm -q
!pip install scikit-learn -q

In [ ]:
import os, random, math
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

import timm
from sklearn.metrics import (
    classification_report, roc_auc_score, cohen_kappa_score, confusion_matrix,
    f1_score, accuracy_score, recall_score, precision_score,
    roc_curve, precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
print(f"PyTorch : {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")

## 2. Configuration

In [ ]:
class CFG:
    # ── Path ──────────────────────────────────────────────────────────────────
    DATA_DIR          = '/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/dataset_preprocessed/dataset_preprocessed'
    CSV_PATH          = '/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/breast-level_annotations.csv'
    OUTPUT_DIR        = '/kaggle/working/outputs'

    # ── Model ─────────────────────────────────────────────────────────────────
    MODEL_NAME        = 'swin_tiny_patch4_window7_224'
    PRETRAINED_WEIGHT = '/kaggle/input/datasets/vickyoktafrian/model-mv-swin-t/swin_tiny_patch4_window7_224.pth'
    IMG_SIZE          = 224
    NUM_CLASSES       = 2                        # CrossEntropy → 2 logits [B, 2]
    CLASS_NAMES       = ['Benign', 'Malignant']
    DROPOUT           = 0.3

    # ── Label mapping ─────────────────────────────────────────────────────────
    BENIGN_BIRADS     = [1, 2]                   # label = 0
    MALIGNANT_BIRADS  = [3, 4, 5]               # label = 1

    # ── Training ──────────────────────────────────────────────────────────────
    EPOCHS            = 100
    BATCH_SIZE        = 16
    NUM_WORKERS       = 4
    WARMUP_EPOCHS     = 5
    LR_BACKBONE       = 1e-5
    LR_HEAD           = 1e-4
    WEIGHT_DECAY      = 1e-4
    PATIENCE          = 15

    # ── Pipeline 3 specifics ──────────────────────────────────────────────────
    # WeightedRandomSampler: setiap sample mendapat bobot = 1 / freq_kelasnya
    # Tujuan: distribusi batch mendekati 50/50 tanpa menambah/kurangi data
    USE_SAMPLER       = True                     # ← AKTIF di P3
    NUM_SAMPLES       = None                     # diisi otomatis = len(train_dataset)

    USE_MULTI_GPU     = True

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device      : {DEVICE}")
print(f"Task        : Benign (BIRADS 1,2) vs Malignant (BIRADS 3,4,5)")
print(f"NUM_CLASSES : {CFG.NUM_CLASSES}  (CrossEntropyLoss — 2 logit output)")
print(f"Loss        : CrossEntropyLoss() — tanpa pos_weight")
print(f"Sampler     : WeightedRandomSampler  ← Pipeline 3")

## 3. Data

In [ ]:
df = pd.read_csv(CFG.CSV_PATH)

# ── Parse breast_birads → label binary ────────────────────────────────────────
def parse_birads(val):
    if pd.isna(val): return None
    digits = ''.join(filter(str.isdigit, str(val).strip()))
    return int(digits) if digits else None

df['birads_num'] = df['breast_birads'].apply(parse_birads)
df = df.dropna(subset=['birads_num'])
df['birads_num'] = df['birads_num'].astype(int)

ALL_BIRADS = CFG.BENIGN_BIRADS + CFG.MALIGNANT_BIRADS
df = df[df['birads_num'].isin(ALL_BIRADS)].copy()

df['label'] = df['birads_num'].apply(lambda x: 1 if x in CFG.MALIGNANT_BIRADS else 0)
df['label'] = df['label'].astype(int)

# ── Distribusi ────────────────────────────────────────────────────────────────
print("=" * 45)
print("Distribusi Benign vs Malignant:")
for i, name in enumerate(CFG.CLASS_NAMES):
    n = (df['label'] == i).sum()
    print(f"  {name:12s} (label={i}): {n:>5} samples ({n/len(df)*100:.1f}%)")
print(f"  {'Total':12s}         : {len(df):>5}")

print("\nBreakdown per BI-RADS:")
for b in sorted(df['birads_num'].unique()):
    n = (df['birads_num'] == b).sum()
    tag = 'Malignant' if b in CFG.MALIGNANT_BIRADS else 'Benign'
    print(f"  BIRADS {b} → {tag:9s}: {n:>5}")
print("=" * 45)

# ── Build CC-MLO pairs ────────────────────────────────────────────────────────
def build_pairs(df):
    pairs = []
    for (sid, lat), grp in df.groupby(['study_id', 'laterality']):
        cc  = grp[grp['view_position'] == 'CC']
        mlo = grp[grp['view_position'] == 'MLO']
        if len(cc) == 0 or len(mlo) == 0: continue
        pairs.append({
            'study_id':     sid,
            'laterality':   lat,
            'cc_image_id':  cc.iloc[0]['image_id'],
            'mlo_image_id': mlo.iloc[0]['image_id'],
            'label':        int(cc.iloc[0]['label']),
            'birads':       int(cc.iloc[0]['birads_num']),
            'split':        cc.iloc[0]['split']
        })
    return pd.DataFrame(pairs)

pairs_df = build_pairs(df)
train_df = pairs_df[pairs_df['split'] == 'training'].reset_index(drop=True)
val_df   = pairs_df[pairs_df['split'] == 'test'].reset_index(drop=True)

if len(val_df) == 0:
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(pairs_df, test_size=0.2,
                                        stratify=pairs_df['label'], random_state=42)
    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)

n_tr_mal = int(train_df['label'].sum())
n_tr_ben = len(train_df) - n_tr_mal
n_vl_mal = int(val_df['label'].sum())
n_vl_ben = len(val_df) - n_vl_mal

print(f"Train pairs : {len(train_df):>4}  | Benign: {n_tr_ben} | Malignant: {n_tr_mal} ({train_df['label'].mean()*100:.1f}% mal)")
print(f"Val   pairs : {len(val_df):>4}  | Benign: {n_vl_ben} | Malignant: {n_vl_mal} ({val_df['label'].mean()*100:.1f}% mal)")

# ── Hitung sample weight untuk WeightedRandomSampler ─────────────────────────
# Bobot tiap kelas = 1 / jumlah sample kelas tersebut
# Sehingga kelas minoritas (malignant) mendapat bobot lebih tinggi
class_counts   = np.array([n_tr_ben, n_tr_mal], dtype=float)
class_weights  = 1.0 / class_counts
print(f"\nClass counts  : Benign={n_tr_ben}, Malignant={n_tr_mal}")
print(f"Class weights : Benign={class_weights[0]:.6f}, Malignant={class_weights[1]:.6f}")
print(f"Weight ratio  : {class_weights[1]/class_weights[0]:.2f}× (malignant lebih berat)")

## 4. Dataset & DataLoader

In [ ]:
def get_image_path(data_dir, study_id, image_id):
    base = Path(data_dir) / str(study_id)
    for ext in ['.png', '.jpg', '.jpeg']:
        p = base / f'{image_id}{ext}'
        if p.exists(): return str(p)
    for ext in ['.png', '.jpg', '.jpeg']:
        p = Path(data_dir) / f'{image_id}{ext}'
        if p.exists(): return str(p)
    return None


def get_transforms(split='train', img_size=224):
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    if split == 'train':
        return T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.2),
            T.RandomRotation(degrees=15),
            T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
            T.ColorJitter(brightness=0.3, contrast=0.3),
            T.ToTensor(),
            T.Normalize(mean=mean, std=std),
        ])
    return T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), T.Normalize(mean, std)])


class VinDrDataset(Dataset):
    def __init__(self, df, data_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.data_dir  = data_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def _load(self, study_id, image_id):
        path = get_image_path(self.data_dir, study_id, image_id)
        if path is None:
            return Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
        return Image.open(path).convert('RGB')

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cc  = self._load(row['study_id'], row['cc_image_id'])
        mlo = self._load(row['study_id'], row['mlo_image_id'])
        if self.transform:
            cc  = self.transform(cc)
            mlo = self.transform(mlo)
        # label sebagai int64 untuk CrossEntropyLoss
        return cc, mlo, torch.tensor(row['label'], dtype=torch.long)


train_dataset = VinDrDataset(train_df, CFG.DATA_DIR, get_transforms('train', CFG.IMG_SIZE))
val_dataset   = VinDrDataset(val_df,   CFG.DATA_DIR, get_transforms('val',   CFG.IMG_SIZE))

# ── Pipeline 3: WeightedRandomSampler ────────────────────────────────────────
# Assign bobot ke setiap sample sesuai kelasnya
sample_weights = class_weights[train_df['label'].values]          # shape: [N]
sample_weights = torch.from_numpy(sample_weights).float()

# num_samples = len(train_dataset) agar epoch punya jumlah iterasi sama seperti tanpa sampler
CFG.NUM_SAMPLES = len(train_dataset)
sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = CFG.NUM_SAMPLES,
    replacement = True                  # replacement=True wajib untuk oversampling
)

# shuffle=False saat pakai sampler (sampler sudah handle urutan)
train_loader = DataLoader(
    train_dataset, batch_size=CFG.BATCH_SIZE,
    sampler=sampler, num_workers=CFG.NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False,
    num_workers=CFG.NUM_WORKERS, pin_memory=True
)

print(f"Train batches : {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"Sampler       : WeightedRandomSampler (num_samples={CFG.NUM_SAMPLES}, replacement=True)")
print(f"Label dtype   : torch.long  (CrossEntropyLoss)")

# Verifikasi distribusi batch setelah sampler
label_counts = {0: 0, 1: 0}
for _, _, lbl in train_loader:
    for l in lbl.numpy():
        label_counts[int(l)] += 1
total_sampled = sum(label_counts.values())
print(f"\nVerifikasi distribusi setelah sampler (1 epoch simulasi):")
print(f"  Benign    : {label_counts[0]:>5} ({label_counts[0]/total_sampled*100:.1f}%)")
print(f"  Malignant : {label_counts[1]:>5} ({label_counts[1]/total_sampled*100:.1f}%)")
print(f"  → Distribusi mendekati 50/50 ✅" if abs(label_counts[0]-label_counts[1]) < total_sampled*0.15 else f"  → Distribusi: cek ulang sampler weight")

## 5. Model — Late Fusion

In [ ]:
class SwinCELateFusion(nn.Module):
    """
    Late-Fusion Swin Transformer — Pipeline 3 (CrossEntropyLoss).

    Architecture:
      CC  ──► Swin Encoder (shared weights) ──► cc_feat  [B, D]
      MLO ──► Swin Encoder (shared weights) ──► mlo_feat [B, D]
                                                     │
                                     torch.cat([cc_feat, mlo_feat])  → [B, 2D]
                                                     │
                                               MLP Classifier
                                                     │
                                              logits [B, 2]  ← 2 kelas (CE)

    Loss : CrossEntropyLoss() — tidak ada pos_weight (balancing via sampler)
    """

    def __init__(self, cfg):
        super().__init__()

        self.encoder = timm.create_model(
            cfg.MODEL_NAME, pretrained=True, num_classes=0, global_pool='avg'
        )
        D = self.encoder.num_features
        print(f"Encoder dim (D)      : {D}")
        print(f"Classifier input dim : {D * 2}  (2×D)")
        print(f"Classifier output    : {cfg.NUM_CLASSES}  (CrossEntropyLoss)")

        self.classifier = nn.Sequential(
            nn.Linear(D * 2, 512),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT / 2),
            nn.Linear(128, cfg.NUM_CLASSES)   # output: [B, 2]
        )

        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, cc: torch.Tensor, mlo: torch.Tensor) -> torch.Tensor:
        cc_feat  = self.encoder(cc)
        mlo_feat = self.encoder(mlo)
        fused    = torch.cat([cc_feat, mlo_feat], dim=1)
        return self.classifier(fused)           # [B, 2]


# ── Instantiate ───────────────────────────────────────────────────────────────
model = SwinCELateFusion(CFG)

# ── Load pretrained weights (encoder only) ────────────────────────────────────
ckpt = torch.load(CFG.PRETRAINED_WEIGHT, map_location='cpu')
sd   = ckpt.get('model', ckpt.get('state_dict', ckpt)) if isinstance(ckpt, dict) else ckpt
ms   = model.encoder.state_dict()
filt = {k: v for k, v in sd.items() if k in ms and v.shape == ms[k].shape}
model.encoder.load_state_dict(filt, strict=False)
print(f"Pretrained   : {len(filt)}/{len(ms)} layers loaded")
assert len(filt) > 100, f"Terlalu sedikit layer loaded ({len(filt)}) — cek file .pth!"

if CFG.USE_MULTI_GPU and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(DEVICE)
print(f"Total params : {sum(p.numel() for p in model.parameters()):,}")

## 6. Loss & Optimizer

In [ ]:
# ── Pipeline 3: CrossEntropyLoss tanpa class weights ─────────────────────────
#
# Kenapa CrossEntropyLoss (bukan BCE)?
#   - CrossEntropyLoss menerima label int [B] dan logit [B, 2]
#   - Lebih natural untuk multi-class, konsisten dengan P4 (Undersampling)
#   - Imbalance sudah ditangani WeightedRandomSampler, tidak perlu loss weight
#
# Kenapa tidak pakai pos_weight di sini?
#   - Karena sampler sudah menyeimbangkan distribusi batch (~50/50)
#   - Menambah weight di loss sekaligus akan double-penalize malignant
#
criterion = nn.CrossEntropyLoss()
print(f"Loss         : CrossEntropyLoss()  (tanpa pos_weight — balance via sampler)")
print(f"Label dtype  : torch.long  (integer class index)")


def get_optimizer(model, cfg, backbone_frozen=True):
    base = model.module if hasattr(model, 'module') else model
    backbone_lr = 0.0 if backbone_frozen else cfg.LR_BACKBONE
    return torch.optim.AdamW([
        {'params': base.encoder.parameters(),    'lr': backbone_lr,  'weight_decay': cfg.WEIGHT_DECAY},
        {'params': base.classifier.parameters(), 'lr': cfg.LR_HEAD,  'weight_decay': cfg.WEIGHT_DECAY},
    ])


def freeze_backbone(model, freeze=True):
    base = model.module if hasattr(model, 'module') else model
    for p in base.encoder.parameters():
        p.requires_grad = not freeze
    print(f"Backbone     : {'FROZEN' if freeze else 'UNFROZEN'}")


freeze_backbone(model, freeze=True)
optimizer = get_optimizer(model, CFG, backbone_frozen=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.EPOCHS, eta_min=1e-7
)
print(f"LR head      : {CFG.LR_HEAD}")
print(f"LR backbone  : {CFG.LR_BACKBONE} (frozen selama {CFG.WARMUP_EPOCHS} epoch pertama)")

## 7. Diagnostic Pre-Training

In [ ]:
print("=" * 60)
print("DIAGNOSTIC — Pipeline 3 (Oversampling: WeightedRandomSampler)")
print("=" * 60)

cc_b, mlo_b, lbl_b = next(iter(train_loader))
cc_b, mlo_b, lbl_b = cc_b.to(DEVICE), mlo_b.to(DEVICE), lbl_b.to(DEVICE)

# [1] CC vs MLO sanity
identical = torch.allclose(cc_b, mlo_b)
print(f"\n[1] CC == MLO pixel-identik : {identical}", '❌ BUG!' if identical else '✅ OK')

# [2] Output shape & range
base = model.module if hasattr(model, 'module') else model
base.eval()
with torch.no_grad():
    logits = model(cc_b, mlo_b)
    probs  = torch.softmax(logits, dim=1)  # [B, 2]
    prob_mal = probs[:, 1]                  # prob kelas Malignant

print(f"\n[2] Output shape  : {logits.shape}  (expected: [{cc_b.size(0)}, 2])")
print(f"    Logit range   : [{logits.min():.3f}, {logits.max():.3f}]")
print(f"    Prob-mal range: [{prob_mal.min():.3f}, {prob_mal.max():.3f}]")
print(f"    Prob-mal mean : {prob_mal.mean():.4f}")

# [3] Batch label distribution setelah sampler
mal_count = (lbl_b == 1).sum().item()
print(f"\n[3] Batch labels (setelah sampler):")
print(f"    Malignant : {int(mal_count)}/{len(lbl_b)} = {mal_count/len(lbl_b):.2%}")
print(f"    Benign    : {len(lbl_b)-int(mal_count)}/{len(lbl_b)} = {(len(lbl_b)-mal_count)/len(lbl_b):.2%}")
print(f"    → Target ~50/50 ✅" if abs(mal_count/len(lbl_b) - 0.5) < 0.2 else "    → Distribusi jauh dari 50/50, cek sampler")

# [4] Loss sanity check
base.train()
with torch.cuda.amp.autocast():
    test_logits = model(cc_b, mlo_b)          # [B, 2]
    loss_ce     = criterion(test_logits, lbl_b)  # lbl_b: torch.long

print(f"\n[4] Init CrossEntropy loss : {loss_ce.item():.4f}")
print(f"    Expected init ≈ ln(2) = {math.log(2):.4f}  (random init 2-class)")
print("=" * 60)
print("✅ Diagnostic OK — siap training!")

## 8. Training

In [ ]:
def find_best_threshold(labels, probs):
    """Sweep threshold 0.05–0.95, pilih yang maximise Macro F1."""
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.05, 0.95, 0.01):
        p  = (probs >= t).astype(int)
        f1 = f1_score(labels, p, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    all_probs, all_labels = [], []

    for cc, mlo, labels in tqdm(loader, desc='Train', leave=False):
        cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(cc, mlo)           # [B, 2]
            loss   = criterion(logits, labels) # labels: torch.long
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        prob_mal = torch.softmax(logits, dim=1)[:, 1]  # prob kelas Malignant
        all_probs.extend(prob_mal.detach().cpu().float().numpy())
        all_labels.extend(labels.cpu().numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    try:    auc = roc_auc_score(labels, probs)
    except: auc = 0.5
    t, f1  = find_best_threshold(labels, probs)
    pos_pm = probs[labels == 1].mean() if (labels == 1).sum() > 0 else 0.0
    neg_pm = probs[labels == 0].mean() if (labels == 0).sum() > 0 else 0.0
    return total_loss / len(loader), auc, f1, pos_pm, neg_pm


@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_probs, all_labels = [], []

    for cc, mlo, labels in tqdm(loader, desc='Val', leave=False):
        cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
        with torch.cuda.amp.autocast():
            logits = model(cc, mlo)            # [B, 2]
            loss   = criterion(logits, labels)
        total_loss += loss.item()
        prob_mal = torch.softmax(logits, dim=1)[:, 1]
        all_probs.extend(prob_mal.detach().cpu().float().numpy())
        all_labels.extend(labels.cpu().numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    try:    auc = roc_auc_score(labels, probs)
    except: auc = 0.5
    t, f1  = find_best_threshold(labels, probs)
    pos_pm = probs[labels == 1].mean() if (labels == 1).sum() > 0 else 0.0
    neg_pm = probs[labels == 0].mean() if (labels == 0).sum() > 0 else 0.0

    preds   = (probs >= t).astype(int)
    ben_acc = accuracy_score(labels[labels==0], preds[labels==0]) if (labels==0).sum() > 0 else 0.0
    mal_acc = accuracy_score(labels[labels==1], preds[labels==1]) if (labels==1).sum() > 0 else 0.0
    print(f"  Benign acc: {ben_acc:.3f}  |  Malignant acc: {mal_acc:.3f}")

    return total_loss / len(loader), auc, f1, probs, labels, t, pos_pm, neg_pm


print("Training functions defined ✅")

In [ ]:
scaler  = torch.cuda.amp.GradScaler()
history = {k: [] for k in ['tr_loss', 'vl_loss', 'tr_auc', 'vl_auc', 'tr_f1', 'vl_f1']}

best_auc          = 0.0
patience_count    = 0
best_probs        = None
best_labels_arr   = None
best_thresh_saved = 0.5

print(f"{'Ep':>3} | {'TrLoss':>7} | {'TrAUC':>6} | {'VlAUC':>6} | {'VlF1':>6} | {'mal_pm':>6} | {'ben_pm':>6} | Status")
print('-' * 85)

for epoch in range(1, CFG.EPOCHS + 1):

    if epoch == CFG.WARMUP_EPOCHS + 1:
        freeze_backbone(model, freeze=False)
        optimizer = get_optimizer(model, CFG, backbone_frozen=False)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG.EPOCHS - CFG.WARMUP_EPOCHS, eta_min=1e-7
        )
        print(f"  → Backbone unfrozen @ epoch {epoch}")

    tr_loss, tr_auc, tr_f1, tr_mal_pm, tr_ben_pm = train_epoch(
        model, train_loader, optimizer, criterion, DEVICE, scaler
    )
    vl_loss, vl_auc, vl_f1, vl_probs, vl_labels, vl_t, vl_mal_pm, vl_ben_pm = val_epoch(
        model, val_loader, criterion, DEVICE
    )
    scheduler.step()

    history['tr_loss'].append(tr_loss); history['vl_loss'].append(vl_loss)
    history['tr_auc'].append(tr_auc);   history['vl_auc'].append(vl_auc)
    history['tr_f1'].append(tr_f1);     history['vl_f1'].append(vl_f1)

    status = ''
    if vl_auc > best_auc:
        best_auc = vl_auc
        patience_count = 0
        base = model.module if hasattr(model, 'module') else model
        torch.save(base.state_dict(), f'{CFG.OUTPUT_DIR}/best_model_p3.pth')
        best_probs, best_labels_arr = vl_probs, vl_labels
        best_thresh_saved = vl_t
        status = '✓ SAVED'
    else:
        patience_count += 1
        if patience_count >= CFG.PATIENCE:
            print(f"\nEarly stopping @ epoch {epoch}")
            break

    sep = ' ←' if vl_mal_pm > vl_ben_pm * 1.1 else ''
    print(f"{epoch:>3} | {tr_loss:>7.4f} | {tr_auc:>6.4f} | {vl_auc:>6.4f} | {vl_f1:>6.4f} | "
          f"{vl_mal_pm:>6.3f} | {vl_ben_pm:>6.3f} | {status}{sep}")

print(f"\n✅ Training selesai. Best Val AUC: {best_auc:.4f}")

## 9. Evaluation

In [ ]:
# ── Load best checkpoint ─────────────────────────────────────────────────────
base = model.module if hasattr(model, 'module') else model
base.load_state_dict(torch.load(f'{CFG.OUTPUT_DIR}/best_model_p3.pth', map_location=DEVICE))

_, _, _, best_probs, best_labels_arr, best_thresh_saved, vl_mal_pm, vl_ben_pm = val_epoch(
    model, val_loader, criterion, DEVICE
)

print(f"Prob mean — Malignant: {vl_mal_pm:.3f} | Benign: {vl_ben_pm:.3f}")
print(f"Best threshold       : {best_thresh_saved:.2f}\n")

# ── Threshold sweep ───────────────────────────────────────────────────────────
print(f"{'Threshold':>10} | {'F1-Macro':>8} | {'Sensitivity':>11} | {'Specificity':>11} | {'Precision':>9}")
print("-" * 60)
for t in np.arange(0.05, 0.96, 0.05):
    p    = (best_probs >= t).astype(int)
    sens = recall_score(best_labels_arr, p, pos_label=1, zero_division=0)
    spec = recall_score(best_labels_arr, p, pos_label=0, zero_division=0)
    f1   = f1_score(best_labels_arr, p, average='macro', zero_division=0)
    prec = precision_score(best_labels_arr, p, pos_label=1, zero_division=0)
    mark = ' ◄' if abs(t - best_thresh_saved) < 0.01 else ''
    print(f"  t={t:.2f}   | {f1:>8.4f} | {sens:>11.3f} | {spec:>11.3f} | {prec:>9.3f}{mark}")

# ── Final metrics ─────────────────────────────────────────────────────────────
final_preds = (best_probs >= best_thresh_saved).astype(int)
auc         = roc_auc_score(best_labels_arr, best_probs)
acc         = accuracy_score(best_labels_arr, final_preds)
f1_macro    = f1_score(best_labels_arr, final_preds, average='macro',    zero_division=0)
f1_weight   = f1_score(best_labels_arr, final_preds, average='weighted', zero_division=0)
kappa       = cohen_kappa_score(best_labels_arr, final_preds)
sensitivity = recall_score(best_labels_arr, final_preds, pos_label=1, zero_division=0)
specificity = recall_score(best_labels_arr, final_preds, pos_label=0, zero_division=0)
precision   = precision_score(best_labels_arr, final_preds, pos_label=1, zero_division=0)

print("\n" + "="*58)
print("    EVALUATION — Pipeline 3 (Oversampling: WeightedRandomSampler)")
print("    Benign (BIRADS 1,2) vs Malignant (BIRADS 3,4,5)")
print("="*58)
print(f"  Loss         : CrossEntropyLoss()  (tanpa pos_weight)")
print(f"  Sampler      : WeightedRandomSampler (replacement=True)")
print(f"  AUC-ROC      : {auc:.4f}")
print(f"  Accuracy     : {acc:.4f}")
print(f"  Macro F1     : {f1_macro:.4f}")
print(f"  Weighted F1  : {f1_weight:.4f}")
print(f"  Sensitivity  : {sensitivity:.4f}  ← recall malignant")
print(f"  Specificity  : {specificity:.4f}  ← recall benign")
print(f"  Precision    : {precision:.4f}  ← precision malignant")
print(f"  Kappa        : {kappa:.4f}")
print(f"  Threshold    : {best_thresh_saved:.2f}")
print("="*58)
print(classification_report(best_labels_arr, final_preds,
      target_names=CFG.CLASS_NAMES, zero_division=0))

## 10. Visualisasi

In [ ]:
fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(2, 3, figure=fig)
fig.suptitle('Pipeline 3 — Oversampling: WeightedRandomSampler + CrossEntropyLoss',
             fontsize=14, fontweight='bold')

# Loss curve
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(history['tr_loss'], label='Train', color='#3498db', lw=2)
ax1.plot(history['vl_loss'], label='Val',   color='#e74c3c', lw=2)
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

# AUC curve
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(history['tr_auc'], label='Train', color='#3498db', lw=2)
ax2.plot(history['vl_auc'], label='Val',   color='#e74c3c', lw=2)
ax2.axhline(y=best_auc, color='green', ls='--', alpha=0.7, label=f'Best={best_auc:.4f}')
ax2.set_title('AUC-ROC'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)

# F1 curve
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(history['tr_f1'], label='Train', color='#3498db', lw=2)
ax3.plot(history['vl_f1'], label='Val',   color='#e74c3c', lw=2)
ax3.set_title('Macro F1'); ax3.set_xlabel('Epoch'); ax3.legend(); ax3.grid(alpha=0.3)

# Confusion Matrix
ax4 = fig.add_subplot(gs[1, 0])
cm = confusion_matrix(best_labels_arr, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=ax4,
            xticklabels=CFG.CLASS_NAMES, yticklabels=CFG.CLASS_NAMES)
ax4.set_title('Confusion Matrix')
ax4.set_xlabel('Predicted'); ax4.set_ylabel('True')

# ROC Curve
ax5 = fig.add_subplot(gs[1, 1])
fpr, tpr, _ = roc_curve(best_labels_arr, best_probs)
ax5.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC = {auc:.4f}')
ax5.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax5.set_title('ROC Curve')
ax5.set_xlabel('FPR (1 − Specificity)'); ax5.set_ylabel('TPR (Sensitivity)')
ax5.legend(); ax5.grid(alpha=0.3)

# Precision-Recall Curve
ax6 = fig.add_subplot(gs[1, 2])
prec_arr, rec_arr, _ = precision_recall_curve(best_labels_arr, best_probs)
ap = average_precision_score(best_labels_arr, best_probs)
ax6.plot(rec_arr, prec_arr, color='#e67e22', lw=2, label=f'AP = {ap:.4f}')
baseline = best_labels_arr.mean()
ax6.axhline(y=baseline, color='gray', ls='--', alpha=0.5, label=f'Baseline = {baseline:.3f}')
ax6.set_title('Precision-Recall (Malignant)')
ax6.set_xlabel('Recall (Sensitivity)'); ax6.set_ylabel('Precision')
ax6.legend(); ax6.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/results_pipeline3_oversampling.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot disimpan → results_pipeline3_oversampling.png")

## 11. Summary — Pipeline 3

In [ ]:
print("="*62)
print("  FINAL SUMMARY — Pipeline 3: Oversampling (WeightedRandomSampler)")
print("="*62)
print(f"  Model       : {CFG.MODEL_NAME}")
print(f"  Task        : Benign (BIRADS 1,2) vs Malignant (BIRADS 3,4,5)")
print(f"  Loss        : CrossEntropyLoss()  (tanpa pos_weight)")
print(f"  Sampler     : WeightedRandomSampler (replacement=True)")
print(f"  num_samples : {CFG.NUM_SAMPLES}  (= len(train_dataset))")
print(f"  Augment     : HFlip, VFlip, Rotation, GaussianBlur, ColorJitter")
print(f"  LR          : backbone={CFG.LR_BACKBONE}, head={CFG.LR_HEAD}")
print(f"  Warmup      : {CFG.WARMUP_EPOCHS} epoch")
print(f"  Threshold   : {best_thresh_saved:.2f}")
print("-"*62)
print(f"  AUC-ROC     : {auc:.4f}")
print(f"  Accuracy    : {acc:.4f}")
print(f"  Macro F1    : {f1_macro:.4f}")
print(f"  Sensitivity : {sensitivity:.4f}  ← recall malignant (prioritas utama)")
print(f"  Specificity : {specificity:.4f}  ← recall benign")
print(f"  Precision   : {precision:.4f}")
print(f"  Kappa       : {kappa:.4f}")
print("="*62)
print("\nCatatan Pipeline 3 vs P2:")
print("  P2: imbalance ditangani di loss (pos_weight)  → data asli masuk")
print("  P3: imbalance ditangani di sampler            → batch ~50/50 otomatis")
print("  P3 efektif karena malignant terlihat lebih sering oleh model per epoch")

metrics_p3 = pd.DataFrame([{
    'pipeline'    : 'P3_Oversampling_WRS',
    'loss'        : 'CrossEntropyLoss',
    'pos_weight'  : None,
    'sampler'     : 'WeightedRandomSampler',
    'auc'         : auc,
    'accuracy'    : acc,
    'macro_f1'    : f1_macro,
    'weighted_f1' : f1_weight,
    'sensitivity' : sensitivity,
    'specificity' : specificity,
    'precision'   : precision,
    'kappa'       : kappa,
    'threshold'   : best_thresh_saved,
}])
metrics_p3.to_csv(f'{CFG.OUTPUT_DIR}/metrics_pipeline3_oversampling.csv', index=False)
print("Saved → metrics_pipeline3_oversampling.csv")